# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 0
%aimport _campaign_lib, api

In [ ]:
import json, os
from _campaign_lib import *

svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

In [ ]:
campaign_config = {
    "queries_per_eval": 15,              # queries per eval step (service default: all)
    "exploration_rate": 0.5,             # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "exclude_steps": ["llm_ranking"],    # steps to skip (e.g. ["entity_profiling"])
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": 3,                 # default: 10
    },
    "eval_llm": {
        "model": "meta-llama/llama-4-maverick-17b-128e-instruct",
        "provider_url": "https://api.groq.com/openai/v1/chat/completions",
        "max_tokens": 4000,
    },
    "grid_search": {
        "context": "A terminology normalization pipeline that matches raw material "
                    "descriptions to standardized database terms using entity profiling "
                    "and candidate ranking.",
        "grid_budget": 35,               # default: 0 (full grid)
        "eval_queries_per_point": 6,     # default: 0 (all queries)
        "shared_queries": False,          # default: True
    },
}

In [ ]:
#@title Pipeline snapshot (full config for reproducibility)
pipeline_config_full = await show_pipeline_snapshot(svc)

In [6]:
#@title Build pipeline params
pipeline_params = configure_pipeline(svc, campaign_config)

## 2. Data

In [7]:
#@title Load datasets
# Set EXCEL_PATH to load from BOM-example.xlsx; leave empty to use stored data
EXCEL_PATH = r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx"  # e.g. "../data/BOM-example.xlsx"
FORCE_RELOAD = False  # Set True to re-read Excel and overwrite stored datasets

train_data, svc["session_terms"] = prepare_datasets(
    svc["store"], svc["backend_id"],
    excel_path=EXCEL_PATH or None,
    force=FORCE_RELOAD,
)


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


In [ ]:
#@title Prepare evaluation context
campaign_rounds = []
baseline_results = []

baseline, eval_data, GROQ_API_KEY, backend_status = await prepare_eval_context(
    svc, train_data,
)

RUN_BASELINE = False  # Set True to evaluate baseline before exploration
if RUN_BASELINE:
    campaign_rounds, baseline_results = await run_baseline_eval(
        baseline, eval_data, campaign_config, svc,
    )

In [10]:
#@title Candidate coverage (post-eval diagnostic)
cov_df = run_coverage_diagnostic(
    baseline_results,
    svc["store"], svc["backend_id"], svc["experiment_id"],
)

Loaded 40 eval queries
Eval runs: 16 completed runs, 3 in-progress
  run_id                name           model                      temp  accuracy  queries
  scan_15a5c1e9         scan                                      0.0   50.0%     6      
  scan_86f17bab         scan                                      0.0   66.7%     6      
  scan_fcb7bf9b         scan                                      0.0   50.0%     6      
  scan_01c3382c         scan                                      0.0   66.7%     6      
  scan_7d0c905a         scan                                      0.0   33.3%     6      
  scan_e9f03615         scan                                      0.0   66.7%     6      
  scan_3aff5881         scan                                      0.0   33.3%     6      
  scan_39b9fc27         scan                                      0.0   50.0%     6      
  scan_dff96710         scan                                      0.0   33.3%     6      
  scan_3dbc066d         scan     

## 3. Explore

Two exploration paths: **Smart Search** (scan advisor + sensitivity scan) or **Grid Search** (brute-force sweep). Use one or both.

### 3a. Smart Search

In [ ]:
#@title Scan advisor
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc, baseline_results, eval_data,
    task_description=TASK_DESCRIPTION if "TASK_DESCRIPTION" in dir() else "",
    coverage_df=cov_df if "cov_df" in dir() else None,
)

In [ ]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_variants = {
    'max_token_candidates': [15, 20, 25],
    'profiling_temperature': [0.0, 0.3, 0.7],
    'profiling_schema': [
        [('-', 'notes'), ('-', 'material_classification')],
        [('+', 'significance', 'string', True, 'Domain relevance of entity')],
        [('~', 'notes', 'industry_sector', 'string', True, 'Industry sector classification')],
    ],
    'relevance_weight_core': [0.5, 0.7, 0.9],
}

scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

In [ ]:
#@title Build diagnostic set (with resume)
(plan_id, search_baseline, diagnostic,
 cached_profiles, variant_library) = await resume_or_build_diagnostic(
    campaign_config, baseline, baseline_results,
    svc, eval_data,
    scan_variants=scan_variants,
)

In [ ]:
#@title Historical data audit & inventory
prompt_index, cached_profiles = audit_historical_data(
    svc["store"], svc["backend_id"],
    diagnostic, cached_profiles,
)

In [ ]:
#@title Coverage advisor
# Knobs: adjust these and re-run to see different strategies
min_queries = 6          # min queries per variant to count as "usable"
axis_requirements = None  # None = require all values; or e.g. {"persona": 2}

coverage = show_scan_coverage(
    search_baseline, variant_library, diagnostic,
    prompt_index,
    pipeline_params=campaign_config.get("pipeline_params"),
    min_queries=min_queries,
    axis_requirements=axis_requirements,
    pipeline_schema=svc.get("pipeline_schema"),
)

In [ ]:
#@title Sensitivity scan
scan_df, axis_profiles = await sensitivity_scan(
    search_baseline, variant_library, diagnostic, svc.get("backend_client"),
    user_focus=campaign_config.get("improvement_areas", ""),
    store=svc["store"], backend_id=svc["backend_id"],
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
    plan_id=plan_id,
    prompt_result_index=prompt_index,
    pipeline_schema=svc.get("pipeline_schema"),
)

In [ ]:
#@title Select scan winner & seed campaign
best_ps, best_params = seed_campaign_from_scan(
    scan_df, axis_profiles, search_baseline, variant_library,
    campaign_rounds, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"], plan_id=plan_id,
)

### 3b. Grid Search

<details>
<summary>Skip if you used Smart Search above.</summary>

Systematic sweep of the prompt configuration space. Maps the accuracy landscape before hill-climbing.

</details>

In [ ]:
#@title Grid campaign overview (existing plans)
merge_plans = False  # Set True to combine results from multiple plans
grid_overview = show_grid_overview(svc, campaign_config, merge_plans=merge_plans)
merged_grid_df = grid_overview.get("merged_grid_df")

In [ ]:
#@title Build or resume grid plan
gs = campaign_config["grid_search"]
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)

(
    grid_plan_id, grid_points, grid_state_lookup,
    grid_axes, layer1_fields, grid_baseline,
) = await resume_or_build_grid(
    campaign_config, baseline, llm_client, llm_model,
    svc["store"], svc["backend_id"],
    improvement_areas=campaign_config.get("improvement_areas", ""),
)

print(f"Grid points: {len(grid_points)}")
print(f"Plan ID: {grid_plan_id}")

In [ ]:
#@title Run grid search
grid_df = await run_grid_search(
    grid_points, grid_state_lookup, eval_data,
    campaign_config["eval_llm"],
    plan_id=grid_plan_id,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_client=svc.get("backend_client"),
    session_terms=svc.get("session_terms"),
    pipeline_params=campaign_config.get("pipeline_params"),
    eval_queries_per_point=gs.get("eval_queries_per_point", 1),
    shared_queries=gs.get("shared_queries", False),
    grid_seed=gs.get("seed", 42),
)

In [ ]:
#@title Display grid results
_display_df = merged_grid_df if merged_grid_df is not None else grid_df
display_grid_results(_display_df, grid_axes, top_k=gs.get("top_k", 5))

In [ ]:
#@title LLM analysis of grid results
_analysis_df = merged_grid_df if merged_grid_df is not None else grid_df
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
grid_analysis = await analyze_grid_results(
    _analysis_df, grid_axes, llm_client, model=llm_model,
)

In [ ]:
#@title Select grid winner and seed campaign
grid_winner = select_and_seed_grid_winner(
    grid_df, merged_grid_df, grid_state_lookup,
    grid_overview.get("plan_dfs", {}), svc, campaign_rounds,
)

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [ ]:
#@title Run optimization (feedback cycle — M3 nodes)
campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    store=svc["store"], backend_id=svc["backend_id"],
    backend_url=svc["backend_client"].base_url,
    pipeline_params=campaign_config.get("pipeline_params"),
    session_terms=svc.get("session_terms"),
)

In [ ]:
#@title Run optimization round (manual)
round_entry = await run_manual_round(
    campaign_rounds, eval_data, campaign_config, svc,
)

## 5. Results

In [ ]:
#@title Campaign comparison table
show_campaign_summary(campaign_rounds)

In [ ]:
#@title Per-query flip tracking (baseline vs final)
show_flip_tracking(campaign_rounds)

In [ ]:
#@title PromptState lineage chain
show_lineage_chain(campaign_rounds)

In [ ]:
#@title Save winner
save_campaign_winner(campaign_rounds, campaign_config, svc["store"], svc["backend_id"])

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config, GROQ_API_KEY)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print("--- SUGGESTED CONFIG (copy to Setup) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)